In [2]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [2]:
## spatial join
# target_features = ?
# join_features = ?
# output_features = os.path.join(gdb, ?)

# fieldmappings = arcpy.FieldMappings()
# fieldmappings.addTable(target_features)
# fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
# sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [3]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [18]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [ ]:
# #====================
# # create 6 runs average
# #====================

# se1 = pd.read_csv(r"E:\Projects\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM\SE_2023_1.csv")
# se2 = pd.read_csv(r"E:\Projects\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM\SE_2023_2.csv")
# se3 = pd.read_csv(r"E:\Projects\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM\SE_2023_3.csv")
# se4 = pd.read_csv(r"E:\Projects\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM\SE_2023_4.csv")
# se5 = pd.read_csv(r"E:\Projects\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM\SE_2023_5.csv")
# se6 = pd.read_csv(r"E:\Projects\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM\SE_2023_6.csv")

# head = se1[[';TAZID', 'CO_TAZID', 'AVGINCOME', 'Enrol_Elem', 'Enrol_Midl', 'Enrol_High', 'CO_FIPS', 'CO_NAME']]

# avg = pd.concat([se1, se2, se3, se4, se5, se6]).groupby(';TAZID', as_index=False)[['TOTHH','HHPOP','RETL','FOOD',
#                                               'MANU','WSLE','OFFI','GVED',
#                                               'HLTH','OTHR','FM_AGRI','FM_MING',
#                                               'FM_CONS','HBJ']].mean()


# avg['RETEMP'] = avg['RETL'] + avg['FOOD']
# avg['OTHEMP'] = avg['OFFI'] + avg['GVED'] + avg['HLTH'] + avg['OTHR']
# avg['INDEMP'] = avg['MANU'] +  avg['WSLE']
# avg['TOTEMP'] = avg['RETEMP'] + avg['INDEMP'] + avg['OTHEMP']
# avg['ALLEMP'] = avg['TOTEMP'] +  avg['FM_AGRI'] + avg['FM_MING'] +  avg['FM_CONS'] + avg['HBJ']
# avg.loc[avg['TOTHH'] > 0, 'HHSIZE'] = avg.HHPOP[avg.TOTHH > 0]/avg.TOTHH[avg.TOTHH > 0]

# new_se = head.merge(avg, on=';TAZID', how='left')
# new_se = new_se[[';TAZID', 'CO_TAZID', 'TOTHH', 'HHPOP', 'HHSIZE', 'TOTEMP', 'RETEMP',
#        'INDEMP', 'OTHEMP', 'ALLEMP', 'RETL', 'FOOD', 'MANU', 'WSLE', 'OFFI',
#        'GVED', 'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING', 'FM_CONS', 'HBJ',
#        'AVGINCOME', 'Enrol_Elem', 'Enrol_Midl', 'Enrol_High', 'CO_FIPS',
#        'CO_NAME']].copy()

# new_se.to_csv(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\Outputs\SE_2023_6Runs_TAZ900.csv", index=False)

In [ ]:
# export to feature class
new_se_postprocessed = pd.read_csv(r"\\wfrcdc1\Volumef\SHARED\Andy\_temp\REMM\SE_2023_6Runs_TAZ900_combinewithBX_split2newTAZ.csv")
new_se_postprocessed.rename({';TAZID':'TAZID'},axis=1, inplace=True)
taz_900 = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\TAZ_900.shp')[['TAZID','SHAPE']].copy()
se2023_sdf = taz_900.merge(new_se_postprocessed, on='TAZID', how='left')
se2023_sdf.rename({'TAZID':'TAZID_V900'},axis=1, inplace=True)
del se2023_sdf['CO_NAME']
del se2023_sdf['CO_FIPS']
del se2023_sdf['CO_TAZID']
se2023_sdf.spatial.to_featureclass(location=os.path.join(gdb, 'se2023_sdf'),sanitize_columns=False) 


# arcpy.analysis.ApportionPolygon(
#     in_features=os.path.join(gdb, 'se2023_sdf'),
#     apportion_fields="TOTHH SUM;HHPOP SUM;TOTEMP SUM;RETEMP SUM;INDEMP SUM;OTHEMP SUM;ALLEMP SUM;RETL SUM;FOOD SUM;MANU SUM;WSLE SUM;OFFI SUM;GVED SUM;HLTH SUM;OTHR SUM;FM_AGRI SUM;FM_MING SUM;FM_CONS SUM;HBJ SUM;AVGINCOME MEAN;Enrol_Elem SUM;Enrol_Midl SUM;Enrol_High SUM;HHSIZE MEAN",
#     target_features="WFv920_TAZ",
#     out_features=r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\Outputs\results.gdb\se_2023_v920",
#     method="AREA",
#     estimation_features=None,
#     weight_field=None,
#     maintain_geometries="MAINTAIN_GEOMETRIES"
# )

# arcpy.analysis.SpatialJoin(
#     target_features="WFv920_TAZ",
#     join_features="se2023_sdf",
#     out_feature_class=r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\Outputs\results.gdb\se_920_900_sj_AVGINCOME",
#     join_operation="JOIN_ONE_TO_ONE",
#     join_type="KEEP_ALL",
#     field_mapping='TAZID "TAZID" true true false 18 Double 0 0,First,#,WFv920_TAZ,TAZID,-1,-1;AVGINCOME "AVGINCOME" true true false 8 BigInteger 0 0,First,#,se2023_sdf,AVGINCOME,-1,-1',
#     match_option="HAVE_THEIR_CENTER_IN",
#     search_radius=None,
#     distance_field_name="",
#     match_fields=None
# )


'e:\\Tasks\\REMM-Manage-Base-Year-Data-2023\\Inputs\\Tables\\Outputs\\scratch.gdb\\se2023_sdf'

In [29]:
#===================================
# combine TAZ apportion  outputs
#=================================
v920_avg_income = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\Outputs\results.gdb\se_920_900_sj_AVGINCOME')
v920se = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\Outputs\results.gdb\se_2023_v920')

v920se['RETEMP'] = v920se['RETL'] + v920se['FOOD']
v920se['OTHEMP'] = v920se['OFFI'] + v920se['GVED'] + v920se['HLTH'] + v920se['OTHR']
v920se['INDEMP'] = v920se['MANU'] +  v920se['WSLE']
v920se['TOTEMP'] = v920se['RETEMP'] + v920se['INDEMP'] + v920se['OTHEMP']
v920se['ALLEMP'] = v920se['TOTEMP'] +  v920se['FM_AGRI'] + v920se['FM_MING'] +  v920se['FM_CONS'] + v920se['HBJ']
v920se.loc[v920se['TOTHH'] > 0, 'HHSIZE'] = v920se.HHPOP[v920se.TOTHH > 0]/v920se.TOTHH[v920se.TOTHH > 0]

v920se = v920se[['TAZID', 'CO_TAZID', 'TOTHH', 'HHPOP', 'HHSIZE', 'TOTEMP', 'RETEMP',
       'INDEMP', 'OTHEMP', 'ALLEMP', 'RETL', 'FOOD', 'MANU', 'WSLE', 'OFFI',
       'GVED', 'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING', 'FM_CONS', 'HBJ'
       , 'Enrol_Elem', 'Enrol_Midl', 'Enrol_High', 'CO_FIPS',
       'CO_NAME']].copy()

v920se = v920se.merge(v920_avg_income, on='TAZID', how='left')

v920se = v920se.rename({'TAZID':';TAZID'},axis=1)
v920se = v920se[[';TAZID', 'CO_TAZID', 'TOTHH', 'HHPOP', 'HHSIZE', 'TOTEMP', 'RETEMP',
       'INDEMP', 'OTHEMP', 'ALLEMP', 'RETL', 'FOOD', 'MANU', 'WSLE', 'OFFI',
       'GVED', 'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING', 'FM_CONS', 'HBJ',
       'AVGINCOME', 'Enrol_Elem', 'Enrol_Midl', 'Enrol_High', 'CO_FIPS',
       'CO_NAME']].copy()

v920se.to_csv(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\Outputs\SE_2023_6Runs_TAZ920.csv", index=False)


In [5]:
base = pd.read_csv(r"E:\Projects\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM\SE_2023.csv")
base.columns

Index([';TAZID', 'CO_TAZID', 'TOTHH', 'HHPOP', 'HHSIZE', 'TOTEMP', 'RETEMP',
       'INDEMP', 'OTHEMP', 'ALLEMP', 'RETL', 'FOOD', 'MANU', 'WSLE', 'OFFI',
       'GVED', 'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING', 'FM_CONS', 'HBJ',
       'AVGINCOME', 'Enrol_Elem', 'Enrol_Midl', 'Enrol_High', 'CO_FIPS',
       'CO_NAME'],
      dtype='object')

In [6]:
# base.groupby('CO_FIPS', as_index=False)[[ 'RETL', 'FOOD', 'MANU', 'WSLE', 'OFFI',
#        'GVED', 'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING', 'FM_CONS', 'HBJ']].sum()

In [7]:
emp_controls = pd.read_csv(r"E:\Projects\REMM-v3.0\data\employment_controls.csv")
emp_controls.columns

Index(['year', 'sector_id', 'number_of_jobs', 'cid'], dtype='object')

In [8]:
emp_controls2 = emp_controls[emp_controls['year'] == 2023]
emp_controls2 = pd.pivot_table(emp_controls2, values='number_of_jobs', index='cid', columns='sector_id', aggfunc='sum', fill_value=0).reset_index()

col_map = {
    9:  "RETL",
    1:  "FOOD",
    5:  "MANU",
    10: "WSLE",
    6:  "OFFI",
    3:  "GVED",
    4:  "HLTH",
    7:  "OTHR",
    11: "FM_AGRI",
    8:  "FM_MING",
    2:  "FM_CONS",
    12: "HBJ"
}

emp_controls2 = emp_controls2.rename(columns=col_map)

emp_controls2.set_index('cid', inplace=True)
emp_controls2 = emp_controls2[['FOOD', 'FM_CONS', 'GVED', 'HLTH', 'MANU', 'OFFI', 'OTHR', 'FM_MING', 'RETL', 'WSLE']]
emp_controls2

sector_id,FOOD,FM_CONS,GVED,HLTH,MANU,OFFI,OTHR,FM_MING,RETL,WSLE
cid,,,,,,,,,,
3,1306,1583,2188,1246,2986,906,3393,48,1989,1132
11,12138,14921,42928,18266,14168,19224,57704,328,21600,13458
35,59441,65628,147864,82044,62950,140439,273641,4485,87734,100675
49,25345,36359,64610,40206,25135,62793,110048,857,48700,20997
57,9446,10498,25922,15620,19126,10592,37298,200,15516,10203


In [20]:
# County Indicators

county_indicators = pd.read_pickle(r"E:\Projects\REMM-v3.0\REMMRun\county_indicators_650_2023.pkl")
county_indicators
county_indicators = county_indicators[['jobs','jobs1', 'jobs2', 'jobs3', 'jobs4', 'jobs5', 'jobs6', 'jobs7', 'jobs8', 'jobs9', 'jobs10', 'job_spaces']]

col_map = {
    'jobs9':  "RETL",
    'jobs1':  "FOOD",
    'jobs5':  "MANU",
    'jobs10': "WSLE",
    'jobs6':  "OFFI",
    'jobs3':  "GVED",
    'jobs4':  "HLTH",
    'jobs7':  "OTHR",
    'jobs11': "FM_AGRI",
    'jobs8':  "FM_MING",
    'jobs2':  "FM_CONS",
    'jobs12': "HBJ"
}

county_indicators = county_indicators.rename(columns=col_map)
county_indicators = county_indicators.reset_index().set_index('county_id')
county_indicators = county_indicators[['FOOD', 'FM_CONS', 'GVED', 'HLTH', 'MANU', 'OFFI', 'OTHR', 'FM_MING', 'RETL', 'WSLE', ]]

county_indicators

,FOOD,FM_CONS,GVED,HLTH,MANU,OFFI,OTHR,FM_MING,RETL,WSLE
county_id,,,,,,,,,,
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11,12138.0,0.0,42928.0,18266.0,14168.0,19224.0,57704.0,0.0,21600.0,13458.0
35,59441.0,0.0,147864.0,82044.0,62950.0,140439.0,273641.0,0.0,87734.0,100675.0
49,25345.0,0.0,64610.0,40206.0,25135.0,62793.0,110048.0,0.0,48700.0,20997.0
57,9446.0,0.0,25922.0,15620.0,19126.0,10592.0,37298.0,0.0,15516.0,10203.0


In [21]:
county_indicators - emp_controls2

,FOOD,FM_CONS,GVED,HLTH,MANU,OFFI,OTHR,FM_MING,RETL,WSLE
county_id,,,,,,,,,,
3,-1306.0,-1583.0,-2188.0,-1246.0,-2986.0,-906.0,-3393.0,-48.0,-1989.0,-1132.0
11,0.0,-14921.0,0.0,0.0,0.0,0.0,0.0,-328.0,0.0,0.0
35,0.0,-65628.0,0.0,0.0,0.0,0.0,0.0,-4485.0,0.0,0.0
49,0.0,-36359.0,0.0,0.0,0.0,0.0,0.0,-857.0,0.0,0.0
57,0.0,-10498.0,0.0,0.0,0.0,0.0,0.0,-200.0,0.0,0.0


In [24]:
# Zone Indicators

zone_indicators = pd.read_pickle(r"E:\Projects\REMM-v3.0\REMMRun\zone_indicators_650_2023.pkl")
zone_indicators = zone_indicators[['COUNTY','jobs','jobs1', 'jobs2', 'jobs3', 'jobs4', 'jobs5', 'jobs6', 'jobs7', 'jobs8', 'jobs9', 'jobs10', 'job_spaces']]

col_map = {
    'jobs9':  "RETL",
    'jobs1':  "FOOD",
    'jobs5':  "MANU",
    'jobs10': "WSLE",
    'jobs6':  "OFFI",
    'jobs3':  "GVED",
    'jobs4':  "HLTH",
    'jobs7':  "OTHR",
    'jobs11': "FM_AGRI",
    'jobs8':  "FM_MING",
    'jobs2':  "FM_CONS",
    'jobs12': "HBJ"
}

zone_indicators = zone_indicators.rename(columns=col_map)
zone_indicators = zone_indicators.groupby(['COUNTY'])[['FOOD', 'FM_CONS', 'GVED', 'HLTH', 'MANU', 'OFFI', 'OTHR', 'FM_MING', 'RETL', 'WSLE', ]].sum()
# zone_indicators = zone_indicators[['FOOD', 'FM_CONS', 'GVED', 'HLTH', 'MANU', 'OFFI', 'OTHR', 'FM_MING', 'RETL', 'WSLE', ]]

zone_indicators

,FOOD,FM_CONS,GVED,HLTH,MANU,OFFI,OTHR,FM_MING,RETL,WSLE
COUNTY,,,,,,,,,,
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11,12138.0,0.0,42928.0,18266.0,14168.0,19224.0,57704.0,0.0,21600.0,13458.0
35,59441.0,0.0,147864.0,82044.0,62950.0,140439.0,273641.0,0.0,87734.0,100675.0
49,25345.0,0.0,64610.0,40206.0,25135.0,62793.0,110048.0,0.0,48700.0,20997.0
57,9446.0,0.0,25922.0,15620.0,19126.0,10592.0,37298.0,0.0,15516.0,10203.0


In [25]:
zone_indicators - emp_controls2

,FOOD,FM_CONS,GVED,HLTH,MANU,OFFI,OTHR,FM_MING,RETL,WSLE
COUNTY,,,,,,,,,,
3,-1306.0,-1583.0,-2188.0,-1246.0,-2986.0,-906.0,-3393.0,-48.0,-1989.0,-1132.0
11,0.0,-14921.0,0.0,0.0,0.0,0.0,0.0,-328.0,0.0,0.0
35,0.0,-65628.0,0.0,0.0,0.0,0.0,0.0,-4485.0,0.0,0.0
49,0.0,-36359.0,0.0,0.0,0.0,0.0,0.0,-857.0,0.0,0.0
57,0.0,-10498.0,0.0,0.0,0.0,0.0,0.0,-200.0,0.0,0.0


In [11]:
jobs = r"E:\Projects\REMM-v3.0\REMMRun\run618year2023jobs.pkl"
jobs = pd.read_pickle(jobs)
result = jobs.groupby(['county_id'], as_index=False)[['sector_id']].count()
result

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\Projects\\REMM-v3.0\\REMMRun\\run618year2023jobs.pkl'

In [ ]:
HHcontrol_path=r"E:\Projects\REMM-v3.0\data\employment_controls.csv"
HH = pd.read_csv(HHcontrol_path)
result = (
    HH[
        (HH['year'] == 2023) &
        (HH['sector_id'].isin([1, 3, 4, 5, 6, 7, 9, 10]))
    ]
    .groupby(['year', 'cid'])['number_of_jobs']
    .sum()
    .reset_index()
)
result.head(5)

In [ ]:
mismatch = jobs[jobs['cid'] != jobs['county_id']]['parcel_id'].unique()

In [ ]:
# taz = pd.DataFrame.spatial.from_featureclass(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\TAZ_900.shp")
# taz.dtypes


In [ ]:

# zone_indicators = pd.read_pickle(r"E:\Projects\REMM-v3.0\REMMRun\zone_indicators_628_2023.pkl")
# del zone_indicators['zone_id']
# zone_indicators.reset_index().head()
# zone_indicators = zone_indicators.fillna(0)



In [ ]:
# jobs_output = pd.read_pickle(r"E:\Projects\REMM-v3.0\REMMRun\run626year2023jobs.pkl")
# jobs_output.head()
# jobs_output = jobs_output[(jobs_output['county_id']==-1) |(jobs_output['cid']==-1)]
# jobs_output.to_csv('unlocated_jobs.csv', index=False)
# jobs_pivot = pd.pivot_table(jobs_output, values='cid', index='parcel_id', columns='sector', aggfunc='count', fill_value=0).reset_index()

# parcels_output = pd.read_pickle(r"E:\Projects\REMM-v3.0\REMMRun\run614year2023parcels.pkl")
# jobs_pivot = jobs_pivot.merge(parcels_output.reset_index()[['parcel_id', 'TAZID_900']], left_on='parcel_id', right_on='parcel_id', how='inner')
# jobs_pivot = jobs_pivot.fillna(0)
# jobs_pivot['parcel_id'] = jobs_pivot['parcel_id'].astype('Int32')
# jobs_pivot['FOOD'] = jobs_pivot['FOOD'].astype('Int32')
# jobs_pivot['GVED'] = jobs_pivot['GVED'].astype('Int32')
# jobs_pivot['HLTH'] = jobs_pivot['HLTH'].astype('Int32')
# jobs_pivot['MANU'] = jobs_pivot['MANU'].astype('Int32')
# jobs_pivot['OFFI'] = jobs_pivot['OFFI'].astype('Int32')
# jobs_pivot['OTHR'] = jobs_pivot['OTHR'].astype('Int32')
# jobs_pivot['RETL'] = jobs_pivot['RETL'].astype('Int32')
# jobs_pivot['WSLE'] = jobs_pivot['WSLE'].astype('Int32')
# jobs_pivot['TAZID_900'] = jobs_pivot['TAZID_900'].astype('Int32')
# # jobs_pivot
# jobs_pivot = taz[['TAZID','SHAPE']].merge(jobs_pivot,left_on='TAZID', right_on='TAZID_900', how='inner')
# jobs_pivot.columns
# # 
# jobs_pivot.spatial.to_featureclass(location=os.path.join(gdb, 'jobs_by_taz900_2023'),sanitize_columns=False)  

In [ ]:
# taz_output = pd.read_csv(r"F:\SHARED\Andy\_temp\REMM\Job_spaces_job_growth_TAZ_12052025.csv")
# taz_output.columns

In [ ]:
# parcels_output = pd.read_pickle(r"E:\Projects\REMM-v3.0\REMMRun\run612year2023parcels.pkl")
# # parcels_output.head()
# parcels_output[parcels_output['zone_id']!= parcels_output['TAZID_900']]

In [ ]:
# taz[['TAZID','SHAPE']].merge(taz_output,left_on='TAZID', right_on='TAZID', how='inner').spatial.to_featureclass(location=os.path.join(gdb, 'full_taz_indicator_2023'),sanitize_columns=False)  

In [ ]:
# taz[['TAZID','SHAPE']].merge(zone_indicators.reset_index(),left_on='TAZID', right_on='zone_id', how='inner').spatial.to_featureclass(location=os.path.join(gdb, 'taz_indicator_2023'),sanitize_columns=False)  

In [ ]:
# jobs[jobs['building_id'] != jobs['parcel_id']].to_csv(os.path.join(outputs,"mismatch_jobs.csv"))

## Add new Spaces to parcels

In [ ]:
# p1 = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels')
# p2 = pd.DataFrame.spatial.from_featureclass(r'e:\\Projects\\REMM-Job-Space-Calculation\\2-Allocate-Spaces\\Outputs\\results.gdb\\parcels_with_job_spaces')

# p1['WFRC_parcel_id'] = p1['WFRC_parcel_id'].astype('Int32')
# p1['MAG_parcel_id'] = p1['MAG_parcel_id'].astype('Int32')
# p1['parcel_id'] = p1['parcel_id'].astype('Int32')

# p2.rename({'wfrc_parcel_id': 'WFRC_parcel_id'},axis=1, inplace=True)
# p2['WFRC_parcel_id'] = p2['WFRC_parcel_id'].astype('Int32')
# p2['MAG_parcel_id'] = p2['MAG_parcel_id'].astype('Int32')
# p2['parcel_id'] = p2['parcel_id'].astype('Int32')

In [ ]:
# wfrc_map = p2.dropna(subset=['WFRC_parcel_id']).set_index('WFRC_parcel_id')['job_spaces'].to_dict()
# mag_map  = p2.dropna(subset=['MAG_parcel_id']).set_index('MAG_parcel_id')['job_spaces'].to_dict()

# # Update p1.job_spaces using WFRC IDs first
# p1['job_spaces'] = p1['WFRC_parcel_id'].map(wfrc_map).combine_first(p1['job_spaces'])

# # Then update using MAG IDs
# p1['job_spaces'] = p1['MAG_parcel_id'].map(mag_map).combine_first(p1['job_spaces'])

# wf_map = p2.dropna(subset=['parcel_id']).set_index('parcel_id')['job_spaces'].to_dict()

# # Update p1.job_spaces using WFRC IDs first
# p1['job_spaces'] = 0
# p1['job_spaces'] = p1['parcel_id'].map(wf_map).combine_first(p1['job_spaces'])


In [ ]:
# output for h5
# p1.drop(['SHAPE', 'OBJECTID'], axis=1).to_csv(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\parcels_20251126.csv",  index=False) 

In [ ]:
# replace current parcels with this if all looks okay
# p1.spatial.to_featureclass(location=r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_NEW',sanitize_columns=False) 
# del p1, p2, wf_map